In [73]:
from citibike.citibike_utils import get_trip_duration_mins
from utils.datetime_utils import timestamp_to_date_col
from pyspark.sql.functions import create_map, lit

In [49]:
from databricks.connect import DatabricksSession
spark = DatabricksSession.builder.serverless().getOrCreate()

In [50]:
df = spark.read.table("citibike_dev.01_bronze.jc_citibike")

In [54]:
from pyspark.sql.functions import col, to_timestamp

df = (
    df
    .withColumn(
        "started_at",
        to_timestamp(col("started_at"), "yyyy-MM-dd HH:mm:ss.SSS")
    )
    .withColumn(
        "ended_at",
        to_timestamp(col("ended_at"), "yyyy-MM-dd HH:mm:ss.SSS")
    )
)

In [55]:
df = get_trip_duration_mins(spark, df, "started_at", "ended_at", "trip_duration_mins")

In [63]:
df = timestamp_to_date_col(spark, df, "started_at", "trip_start_date")

In [64]:
df = df.withColumn("metadata",
    create_map(
    lit("pipeline_id"), lit("placeholder"), 
    lit("run_id"), lit("placeholder"), 
    lit("task_id"), lit("placeholder"), 
    lit("processed_timestamp"), lit("placeholder")
))

In [65]:
df =  df.select(
"ride_id",
"trip_start_date",
"started_at",
"ended_at",
"start_station_name",
"end_station_name", "trip_duration_mins", "metadata" )

In [67]:
df.write.\
mode("overwrite").\
option("overwriteSchema", "true").\
saveAsTable("citibike_dev.02_silver.jc_citibike")